In [ ]:

from google.colab import drive
drive.mount('/content/drive')

puty = "/content/drive/MyDrive/mashob/laba2"
train_path = f"{puty}/train.csv"
test_path = f"{puty}/test.csv"
sample_submission_path = f"{puty}/sample_submission.csv"


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor


train_df = pd.read_csv(train_path, header=0)
test_df = pd.read_csv(test_path, header=0)



n_desc = train_df.shape[1] - 3  # id, SMILES, Tm
desc_cols = [f"Group {i}" for i in range(1, n_desc + 1)]

train_df.columns = ["id", "SMILES", "Tm"] + desc_cols
test_df.columns = ["id", "SMILES"] + desc_cols


for col in desc_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')


# Выбираем признаки и целевую переменную
X = train_df[desc_cols]
y = train_df["Tm"]


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=4
)


lgbm = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.007,
    max_depth=6,
    num_leaves=34,
    subsample=0.85,
    colsample_bytree=0.85,
    subsample_freq=1,
    min_child_samples=20,
    min_child_weight=0.001,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=123,
    n_jobs=-1,
    verbosity=-1
)


print("Обучение модели..")
lgbm.fit(X, y)

# Валидация
val_pred = lgbm.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
print(f"Validation RMSE: {rmse:.4f}")


X_test = test_df[desc_cols]
test_pred = lgbm.predict(X_test)

# Загрузка шаблона и сохранение результата
submission = pd.read_csv(sample_submission_path)
submission["prediction"] = test_pred

output_path = f"{puty}/submission.csv"
submission.to_csv(output_path, index=False)
print(f" Submission сохранён: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Обучение модели...
Validation RMSE: 47.2648
 Submission сохранён: /content/drive/MyDrive/mashob/laba2/submission.csv
